In [23]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# creating spark session
spark = SparkSession.builder \
    .appName("Week5_Spark_Assignment") \
    .getOrCreate()

print("Spark connected successfully!")

Spark connected successfully!


In [24]:
# loading superstore dataset
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

# basic information
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

# schema
df.printSchema()

# sample records
df.show(5)

Total Rows: 9994
Total Columns: 21
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+------------

In [25]:
# Q3: removing duplicates using customer id and order date
df2 = df.dropDuplicates(["Customer ID", "Order Date"])

# Q12: removing rows where customer id is null
df3 = df2.filter(
    F.col("Customer ID").isNotNull()
)

# removing blank customer names
df4 = df3.filter(
    F.col("Customer Name") != ""
)

# Q5: filling missing region values
df_cleaned = df4.na.fill({
    "Region": "Unknown"
})

print("Rows after cleaning:", df_cleaned.count())

Rows after cleaning: 4992


In [26]:
# converting order date to timestamp
df_with_type = df_cleaned.withColumn(
    "Order Date",
    F.to_timestamp("Order Date")
)

# renaming order date column
df_final_clean = df_with_type.withColumnRenamed(
    "Order Date",
    "event_time"
)

# converting quantity to integer
df_final_clean = df_final_clean.withColumn(
    "Quantity",
    F.col("Quantity").cast("int")
)

# safely converting sales to double
df_final_clean = df_final_clean.withColumn(
    "Sales",
    F.expr("try_cast(Sales as double)")
)

# removing invalid sales records
df_final_clean = df_final_clean.filter(
    F.col("Sales").isNotNull()
)

df_final_clean.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = false)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [27]:
print("----- Q4: Average Sales for West Region by Category -----")

west_df = df_final_clean.filter(
    F.col("Region") == "West"
)

west_avg = west_df.groupBy("Category").agg(
    F.avg("Sales").alias("avg_sales")
)

west_avg.show()


print("----- Q6: Cities with More Than 10 Orders -----")

city_groups = df_final_clean.groupBy("City").agg(
    F.count("*").alias("total_orders")
)

city_filtered = city_groups.filter(
    F.col("total_orders") > 10
)

city_filtered.show()


print("----- Q8: Corporate Customers with Quantity >= 5 -----")

corp_users = df_final_clean.filter(
    (F.col("Segment") == "Corporate") &
    (F.col("Quantity") >= 5)
)

corp_users.show(5)


print("----- Q13: Sales Summary -----")

sales_summary = df_final_clean.agg(
    F.min("Sales").alias("min_sales"),
    F.max("Sales").alias("max_sales"),
    F.avg("Sales").alias("avg_sales")
)

sales_summary.show()

----- Q4: Average Sales for West Region by Category -----
+---------------+------------------+
|       Category|         avg_sales|
+---------------+------------------+
|Office Supplies|102.14556873614184|
|      Furniture| 343.2295197368419|
|     Technology|383.11840225563935|
+---------------+------------------+

----- Q6: Cities with More Than 10 Orders -----
+-------------+------------+
|         City|total_orders|
+-------------+------------+
|  Springfield|          71|
|      Phoenix|          29|
|       Monroe|          12|
|        Omaha|          14|
|       Marion|          11|
|     Franklin|          20|
|       Dallas|          77|
|    Oceanside|          13|
|      Oakland|          13|
|  San Antonio|          27|
|      Raleigh|          13|
| Philadelphia|         247|
|   Louisville|          29|
| Fayetteville|          20|
|  Los Angeles|         375|
|    Fairfield|          22|
| Indianapolis|          12|
|    Cleveland|          21|
|San Francisco|         2

In [28]:
# removing any remaining duplicates
step1 = df_final_clean.dropDuplicates()

# filling null sales values with 0
step2 = step1.na.fill({
    "Sales": 0
})

# calculating total revenue by region
final_output = step2.groupBy("Region").agg(
    F.sum("Sales").alias("total_revenue")
)

print("Final Summary Table")

final_output.show()

# saving result
final_output.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("results_summary.csv")

print("Result saved successfully")

Final Summary Table
+-------+------------------+
| Region|     total_revenue|
+-------+------------------+
|  South| 189878.4430000001|
|Central|244500.46719999978|
|   East| 324064.9430000001|
|   West| 324472.0154999999|
+-------+------------------+

Result saved successfully
